# Thư viện và Dữ liệu

In [ ]:
# thư viện
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# nhập kết quả dự báo từ các mô hình
df = pd.concat(map(pd.read_csv, ['predictions_prophet.csv',
                                  'predictions_sarimax.csv',
                                  'predictions_gbm.csv']),
               axis=1)
df.head()

In [ ]:
# chỉ lấy các cột cần thiết
df = df[["ds", "prophet", "sarimax", "gbm_proba"]]
df.head(2)

In [ ]:
# đặt index
df.index = df.pop('ds')
df.head(1)

# Ensemble

In [ ]:
# lấy sai số của từng mô hình
error_prophet = float(pd.read_csv("../best_params_prophet.csv").iloc[4, 1])
error_sarimax = float(pd.read_csv("../best_params_sarimax.csv").iloc[6, 1])
error_gbm     = 1 - float(pd.read_csv("../best_params_classifier.csv").iloc[0, 1])  # 1 - AUC

In [ ]:
# sai số trung bình
average_error = (error_prophet + error_sarimax + error_gbm) / 3
print(f"Sai số trung bình: {average_error}")

In [ ]:
# trọng số ban đầu (mô hình tốt hơn → sai số thấp hơn → trọng số cao hơn)
weight_prophet = (1/3) / (error_prophet / average_error)
print(f"Trọng số Prophet : {weight_prophet:.4f}")

weight_sarimax = (1/3) / (error_sarimax / average_error)
print(f"Trọng số SARIMAX : {weight_sarimax:.4f}")

weight_gbm = (1/3) / (error_gbm / average_error)
print(f"Trọng số GBM     : {weight_gbm:.4f}")

In [ ]:
# tổng trọng số
extra_weight = weight_prophet + weight_sarimax + weight_gbm
print(f"Tổng trọng số: {extra_weight}")

# Dự báo Ensemble

In [ ]:
# ensemble có trọng số
df['ensemble'] = (df.prophet   * weight_prophet +
                  df.sarimax   * weight_sarimax  +
                  df.gbm_proba * weight_gbm) / extra_weight

In [ ]:
# trực quan hóa
df.plot(figsize=(16, 6), legend=True,
        title='Dự báo Ensemble — Q2/2026');